# 02. 베이스라인 + 선형 모델

베이스라인을 먼저 세우고 `Ridge`/`ElasticNet`이 그것을 넘는지 본다.

베이스라인이 중요한 이유: `free_views_1_10` 하나짜리 로그-로그 회귀가 이미 상당히 잘 맞는다.
모델이 이걸 유의미하게 못 이기면 피처나 데이터를 의심해야 한다.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 한글 라벨이 깨지지 않도록 (Windows 기본 폰트)
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (9, 4)


In [2]:
from sklearn.linear_model import ElasticNet, Ridge

from service.model_training import (
    build_pipeline, cross_validate_model, evaluate, load_dataset,
    median_baseline, single_feature_baseline, split_xy, stratified_split,
)
from service.schema import NUMERIC_FEATURE_COLUMNS

X, y = split_xy(load_dataset("labeled"))
split = stratified_split(X, y)
print(f"train {len(split.X_train):,} / test {len(split.X_test):,}")

train 7,192 / test 1,798


## 베이스라인

In [3]:
results = {}
results["중앙값"] = evaluate(split.y_test, median_baseline(split.y_train, len(split.y_test)))
results["free_views 단일회귀"] = evaluate(
    split.y_test, single_feature_baseline(split.X_train, split.y_train, split.X_test)
)
pd.DataFrame(results).T.round(4)

,log_rmse,log_r2,mae,mdape,spearman
중앙값,2.1967,-0.0201,222942.5545,87.1890,NaN
free_views 단일회귀,1.1114,0.7389,168912.5880,63.6426,0.8518


중앙값 베이스라인은 예측이 상수라 Spearman이 정의되지 않는다(NaN).

단일회귀는 logR² 0.74로 이미 꽤 높다 — 앞 10화 조회수가 구매수의 주된 설명 변수다.

## 선형 모델

수치 피처가 3~7자릿수에 걸쳐 치우쳐 있어 **log1p를 거치지 않으면 선형 모델이 자기
단일회귀 베이스라인보다도 나쁘다**. `impute_log_scale=True`가 중앙값 대치 → log1p → 표준화를 붙인다.

In [4]:
for name, estimator in [("ridge", Ridge(alpha=1.0)),
                        ("elasticnet", ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000))]:
    model = build_pipeline(estimator, impute_log_scale=True)
    cv = cross_validate_model(model, split.X_train, split.y_train)
    model.fit(split.X_train, split.y_train)
    results[name] = {**evaluate(split.y_test, model.predict(split.X_test)), **cv}

pd.DataFrame(results).T[["log_rmse", "log_r2", "mdape", "spearman", "cv_log_rmse"]].round(4)

,log_rmse,log_r2,mdape,spearman,cv_log_rmse
중앙값,2.1967,-0.0201,87.1890,NaN,NaN
free_views 단일회귀,1.1114,0.7389,63.6426,0.8518,NaN
ridge,1.0478,0.7679,60.5340,0.8727,1.0452
elasticnet,1.0482,0.7678,59.5544,0.8777,1.0480


### log1p 없이 돌리면 어떻게 되는지 (대조)

전처리 하나가 결론을 뒤집는다는 걸 남겨둔다.

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from service.model_training import genre_tokens
from service.schema import CATEGORICAL_FEATURE_COLUMNS

no_log = TransformedTargetRegressor(
    regressor=Pipeline([
        ("prep", ColumnTransformer([
            ("genres", CountVectorizer(analyzer=genre_tokens, binary=True), CATEGORICAL_FEATURE_COLUMNS[0]),
            ("numeric", Pipeline([("impute", SimpleImputer(strategy="median")),
                                  ("scale", StandardScaler())]), NUMERIC_FEATURE_COLUMNS),
        ])),
        ("model", Ridge(alpha=1.0)),
    ]),
    func=np.log1p, inverse_func=np.expm1,
)
no_log.fit(split.X_train, split.y_train)
comparison = pd.DataFrame({
    "ridge (log1p 있음)": results["ridge"],
    "ridge (log1p 없음)": evaluate(split.y_test, no_log.predict(split.X_test)),
    "단일회귀 베이스라인": results["free_views 단일회귀"],
}).T[["log_rmse", "log_r2", "spearman"]]
comparison.round(4)

,log_rmse,log_r2,spearman
ridge (log1p 있음),1.0478,0.7679,0.8727
ridge (log1p 없음),1.2263,0.6821,0.8362
단일회귀 베이스라인,1.1114,0.7389,0.8518


log1p 없이는 Ridge가 단일회귀 베이스라인에 진다. 선형 모델을 "못 쓴다"고 결론 내리기 전에
전처리부터 확인해야 한다는 사례.

다음: 트리 앙상블(`03_tree_ensembles.ipynb`).